# Bridged T-coil

[Palace](https://awslabs.github.io/palace/) is an open-source 3D electromagnetic simulator supporting eigenmode, driven (S-parameter), and electrostatic simulations. This notebook uses `gsim` and Palace to design a **bridged T-coil**: two coupled inductors with a center tap and a **bridge capacitance**, used to drive a **capacitive load** over a wider bandwidth while keeping the input matched to 50 Ω. The circuit targets come from Razavi, *The Bridged T-Coil*, IEEE Solid-State Circuits Magazine 7(4), 2015. A recent application is the equalization network between a CMOS driver and a silicon modulator in Li et al., *Nature Electronics* 6, 2023.

**Requirements:**

- IHP PDK: `uv pip install ihp-gdsfactory`
- [GDSFactory+](https://gdsfactory.com) account for cloud simulation

**Runtime.** The baseline layout is one cloud simulation, the layout sweep adds 9 more (the evaluation budget), and the best design is re-solved once with a finer mesh. 

**Goal and scope.** This is the first step of [gsim#275](https://github.com/gdsfactory/gsim/issues/275). The idea is to go from circuit targets to a layout: pick a load, compute the ideal $L_1 = L_2$, coupling $k$ and bridge capacitance $C_B$, build a parameterized T-coil on the IHP top metals, simulate it, and check whether the layout meets the return-loss, bandwidth and area targets. The targets used here (50 Ω, 100 fF load, maximally flat response) are example values, not a specific product. The search is a small fixed grid, not a full optimizer. Shield and fill variants and process variations are left as next steps at the end.

In [ ]:
import gdsfactory as gf
import matplotlib.pyplot as plt
import numpy as np
from gdsfactory import Component
from gdsfactory.typings import LayerSpec
from ihp import PDK

PDK.activate()

## 1. Circuit targets

The T-coil sits between a source with resistance $R$ (node 1), a termination $R_T = R$ (node 2) and a load capacitance $C_L$ at the center tap (node 3). Choosing $L_1 = L_2 = L$ and the pole-zero cancellation conditions makes $Z_\text{in} = R_T$ at **all** frequencies and reduces the transfer to the load to a second-order response with damping factor $\zeta$. Given $\zeta$:

$$
L = \frac{R^2 C_L}{4}\left(1 + \frac{1}{4\zeta^2}\right), \qquad
k = \frac{4\zeta^2 - 1}{4\zeta^2 + 1}, \qquad
C_B = \frac{C_L}{16\,\zeta^2}.
$$

With $\zeta = \sqrt{2}/2$ (maximally flat), $k = 1/3$, $C_B = C_L/8$ and the bandwidth is $2.83/(RC_L)$. The load $C_L$ below stands for a modulator electrode or ESD/pad capacitance; change it to your actual load.

In [ ]:
R_T = 50.0  # termination / source resistance [Ohm]
C_L = 100e-15  # load capacitance at the center tap [F]
ZETA = np.sqrt(2) / 2  # damping factor (maximally flat)


def razavi_targets(R, C_L, zeta=np.sqrt(2) / 2):
    """Ideal bridged T-coil values (Razavi 2015)."""
    L = R**2 * C_L / 4 * (1 + 1 / (4 * zeta**2))
    k = (4 * zeta**2 - 1) / (4 * zeta**2 + 1)
    C_B = C_L / (16 * zeta**2)
    return {"L1": L, "L2": L, "k": k, "M": k * L, "C_B": C_B}


targets = razavi_targets(R_T, C_L, ZETA)
print(
    f"L1 = L2 = {targets['L1'] * 1e12:.1f} pH, k = {targets['k']:.3f}, "
    f"M = {targets['M'] * 1e12:.1f} pH, C_B = {targets['C_B'] * 1e15:.2f} fF"
)
print(
    f"total coil inductance L1 + L2 + 2M = {(2 * targets['L1'] + 2 * targets['M']) * 1e12:.0f} pH"
)

### Lumped T-coil model

The same 3-node model is used for the ideal targets and, later, to fit the EM data. Node order is (P1, P2, P3) = (input, termination, center tap), all referenced to ground:

- $L_1$ between P1 and P3, $L_2$ between P3 and P2, aiding coupling $M = k\sqrt{L_1 L_2}$ (current flowing P1 → P3 → P2 sees $L_1 + L_2 + 2M$), with series losses $R_1, R_2$;
- bridge capacitance $C_B$ between P1 and P2;
- parasitic shunt capacitances $C_A, C_{Bg}, C_T$ from each node to ground.

With the inductor branch voltages $(V_1 - V_3, V_2 - V_3)$ the branch impedance matrix is $\begin{pmatrix} R_1 + j\omega L_1 & -j\omega M \\ -j\omega M & R_2 + j\omega L_2\end{pmatrix}$ (both branch currents flow *toward* the tap, hence the minus sign), and the nodal admittance is $Y = T^\top Z_L^{-1} T$ plus the capacitive stamps.

In [ ]:
def tcoil_y(f, L1, L2, k, C_B, R1=0.0, R2=0.0, C_A=0.0, C_Bg=0.0, C_T=0.0):
    """3-node admittance matrix (P1, P2, P3 to ground) of a lumped T-coil."""
    w = 2 * np.pi * np.asarray(f, dtype=float)
    M = k * np.sqrt(L1 * L2)
    Zl = np.zeros((len(w), 2, 2), complex)
    Zl[:, 0, 0] = R1 + 1j * w * L1
    Zl[:, 1, 1] = R2 + 1j * w * L2
    Zl[:, 0, 1] = Zl[:, 1, 0] = -1j * w * M
    T = np.array([[1, 0, -1], [0, 1, -1]], float)
    Y = T.T @ np.linalg.inv(Zl) @ T
    jw = 1j * w
    Y[:, 0, 0] += jw * (C_B + C_A)
    Y[:, 1, 1] += jw * (C_B + C_Bg)
    Y[:, 0, 1] -= jw * C_B
    Y[:, 1, 0] -= jw * C_B
    Y[:, 2, 2] += jw * C_T
    return Y


def y_to_s(Y, z0=50.0):
    """Admittance to scattering matrix, same reference z0 on every port."""
    eye = np.eye(Y.shape[-1])
    return np.linalg.solve(eye + z0 * Y, eye - z0 * Y)


def terminated_response(S, f, C_L, z0=50.0):
    """P1 driven from a z0 source, P2 terminated in z0, P3 loaded by C_L.

    The 3-port S-matrix already terminates P2 in z0; P3 is closed with the
    reflection of C_L. Returns the input reflection at P1 and the voltage
    transfer H = V3 / (V_src / 2) to the load, normalized to 1 at DC.
    """
    w = 2 * np.pi * np.asarray(f)
    ZL = 1 / (1j * w * C_L)
    G = (ZL - z0) / (ZL + z0)
    S11 = S[:, 0, 0] + S[:, 0, 2] * G * S[:, 2, 0] / (1 - S[:, 2, 2] * G)
    H = S[:, 2, 0] / (1 - S[:, 2, 2] * G) * (1 + G)
    return S11, H


def bandwidth(f, H, drop_db=3.0):
    """-3 dB frequency of |H|, linearly interpolated between grid points."""
    mag = 20 * np.log10(np.abs(H) / np.abs(H[0]))
    below = np.nonzero(mag < -drop_db)[0]
    if not len(below):
        return float(f[-1])
    i = below[0]
    return float(np.interp(-drop_db, [mag[i], mag[i - 1]], [f[i], f[i - 1]]))

Sanity check against the theory: the ideal T-coil gives $S_{11} \equiv 0$ and a bandwidth of $2.83/(2\pi R C_L)$, compared with $1/(2\pi (R\,\|\,R_T) C_L)$ when $C_L$ sits directly on the terminated line.

In [ ]:
f_ideal = np.linspace(0.1e9, 200e9, 2000)
S_ideal = y_to_s(
    tcoil_y(f_ideal, targets["L1"], targets["L2"], targets["k"], targets["C_B"])
)
S11_ideal, H_ideal = terminated_response(S_ideal, f_ideal, C_L)
H_bare = 1 / (1 + 1j * 2 * np.pi * f_ideal * (R_T / 2) * C_L)

bw_ideal, bw_bare = bandwidth(f_ideal, H_ideal), bandwidth(f_ideal, H_bare)
print(f"max |S11| ideal T-coil: {np.abs(S11_ideal).max():.1e}")
print(
    f"BW ideal T-coil: {bw_ideal / 1e9:.1f} GHz (theory {2.83 / (R_T * C_L) / (2 * np.pi) / 1e9:.1f})"
)
print(f"BW without T-coil: {bw_bare / 1e9:.1f} GHz")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(f_ideal / 1e9, 20 * np.log10(np.abs(H_ideal)), label="Ideal T-coil")
ax.plot(f_ideal / 1e9, 20 * np.log10(np.abs(H_bare)), "--", label="No T-coil")
ax.axhline(-3, color="gray", lw=0.8)
ax.set(
    xlabel="Frequency (GHz)",
    ylabel="|H| (dB)",
    title="Voltage transfer to C_L",
    ylim=(-15, 2),
)
ax.grid(True)
ax.legend()
plt.tight_layout()
plt.show()

## 2. Parameterized T-coil layout

A planar symmetric two-turn coil whose electrical midpoint is brought out as the center tap:

- **L1**: P1 → left half of the outer turn → TopMetal2 bridge → right half of the inner turn → tap;
- **L2**: P2 → right half of the outer turn → TopMetal1 underpass (TopVia2 at both ends) → left half of the inner turn → tap.

Both halves are wound in the same sense, so the coupling is aiding as the T-coil requires. The midpoint lies at the bottom of the inner turn, so the tap lead (P3) leaves straight down between the P1/P2 leads with no extra underpass. $L$ is set mainly by the outer `diameter`; $k$ by `width` and `space` (tighter turns → stronger coupling). The interwinding and crossover capacitance provides part of $C_B$.

Geometry constraints from the IHP rule deck (`sg13g2_tech_default.json`, `ihp-gdsfactory==0.2.8`): TM2 min width/space 2.0 µm (space ≥ 5 µm for lines wider than 5 µm), TM1 min width/space 1.64 µm, TopVia2 0.9 µm with 0.5 µm enclosure.

In [ ]:
DRC = {
    "TM2_min_width": 2.0,
    "TM2_min_space": 2.0,
    "TM2_wide_space": 5.0,  # space for lines wider than 5 um (TM2.bR)
    "TM1_min_width": 1.64,
    "TV2_size_plus_enclosure": 0.9 + 2 * 0.5,
}


def check_tcoil_constraints(diameter, width, space, max_area_um2=None):
    """Return a list of violated layout constraints (empty if OK)."""
    errors = []
    if width < DRC["TM2_min_width"]:
        errors.append(f"width {width} < TM2 min width {DRC['TM2_min_width']}")
    if width < DRC["TV2_size_plus_enclosure"]:
        errors.append(f"width {width} too narrow for a TopVia2 + enclosure")
    min_space = DRC["TM2_wide_space"] if width > 5.0 else DRC["TM2_min_space"]
    if space < min_space:
        errors.append(f"space {space} < TM2 min space {min_space} for width {width}")
    p = width + space
    if diameter < 2 * (1.5 * p + 1.5 * width + p) + width:
        errors.append(f"diameter {diameter} too small for width/space {width}/{space}")
    if max_area_um2 is not None and diameter**2 > max_area_um2:
        errors.append(f"area {diameter**2:.0f} um2 > limit {max_area_um2:.0f} um2")
    return errors


@gf.cell
def tcoil(
    diameter: float = 60.0,
    width: float = 3.0,
    space: float = 2.0,
    lead_length: float = 15.0,
    layer_metal: LayerSpec = "TopMetal2drawing",
    layer_metal_name: str = "TopMetal2",
    layer_underpass: LayerSpec = "TopMetal1drawing",
    layer_underpass_name: str = "TopMetal1",
    layer_inductor: LayerSpec = "INDdrawing",
    layer_no_fill: LayerSpec = "NoMetFillerdrawing",
) -> Component:
    """Symmetric 2-turn center-tapped T-coil (square, 45-degree crossover).

    Args:
        diameter: outer side length of the coil (um).
        width: trace width (um).
        space: spacing between turns (um).
        lead_length: length of the three feed leads (um).
        layer_metal: main winding layer.
        layer_metal_name: via_stack name of the main winding layer.
        layer_underpass: crossover underpass layer.
        layer_underpass_name: via_stack name of the underpass layer.
        layer_inductor: inductor recognition marker.
        layer_no_fill: metal-fill blocking marker.

    Returns:
        Component with ports P1 (input), P2 (termination), P3 (center tap),
        all facing south on ``layer_metal``.
    """
    errors = check_tcoil_constraints(diameter, width, space)
    if errors:
        raise ValueError("; ".join(errors))

    w, s = width, space
    p = w + s  # turn pitch
    a0 = diameter / 2 - w / 2  # outer turn centerline half-size
    a1 = a0 - p  # inner turn centerline half-size
    x0 = w + s  # x of the P1/P2 leads (tap lead at x = 0)
    c = 1.5 * p  # half-length of the crossover region
    y_end = -a0 - lead_length

    comp = gf.Component()

    def wire(points, layer):
        comp.add_ref(gf.path.extrude(gf.Path(points), layer=layer, width=w))

    # L1: one continuous TopMetal2 path, bridge included
    wire(
        [
            (-x0, y_end),
            (-x0, -a0),
            (-a0, -a0),
            (-a0, a0),
            (-c, a0),
            (-p / 2, a0),
            (p / 2, a1),
            (c, a1),
            (a1, a1),
            (a1, -a1),
            (0, -a1),
        ],
        layer_metal,
    )
    # L2: outer half (TM2), underpass (TM1), inner half (TM2)
    wire([(x0, y_end), (x0, -a0), (a0, -a0), (a0, a0), (c, a0)], layer_metal)
    wire([(c, a0), (p / 2, a0), (-p / 2, a1), (-c, a1)], layer_underpass)
    wire([(-c, a1), (-a1, a1), (-a1, -a1), (0, -a1)], layer_metal)
    # Center-tap lead
    wire([(0, -a1), (0, y_end)], layer_metal)

    # TopMetal1 <-> TopMetal2 transitions at both underpass ends
    via = gf.get_component(
        "via_stack",
        bottom_layer=layer_underpass_name,
        top_layer=layer_metal_name,
        size=(w, w),
    )
    for xy in ((c, a0), (-c, a1)):
        ref = comp.add_ref(via)
        ref.move(ref.center, xy)

    # Inductor recognition + no-fill markers over the coil body
    half = diameter / 2 + s
    for layer in (layer_inductor, layer_no_fill):
        comp.add_polygon(
            [(-half, -half), (-half, half), (half, half), (half, -half)], layer=layer
        )

    for name, x in (("P1", -x0), ("P2", x0), ("P3", 0.0)):
        comp.add_port(
            name=name, center=(x, y_end), width=w, orientation=270, layer=layer_metal
        )

    comp.info["area_um2"] = diameter**2
    comp.flatten()
    return comp


coil = tcoil()
coil.draw_ports()
coil.plot()

### Ground reference: Metal1 frame (and optional slotted shield)

As in the inductor notebook, the lumped ports need a nearby return conductor. A Metal1 frame surrounds the coil, with its bottom wall running under the three lead ends so that each  interlayer` port spans Metal1 → TopMetal2. A frame rather than a solid plane avoids shorting the magnetic field of the coil. `shield=True` adds a slotted Metal1 ground shield under the
coil (two combs tied to opposite walls, no closed loops) for the shield-variant comparison.

In [ ]:
def add_ground_frame(
    coil: Component,
    margin: float = 20.0,
    wall: float = 10.0,
    layer: LayerSpec = "Metal1drawing",
    shield: bool = False,
    shield_width: float = 2.0,
    shield_space: float = 2.0,
) -> Component:
    """Metal1 reference frame around the coil, optional slotted shield."""
    c = gf.Component()
    ref = c.add_ref(coil)
    c.add_ports(ref.ports)
    bb = coil.dbbox()
    yb = min(p.center[1] for p in coil.ports)  # bottom wall starts at the ports
    xl, xr = bb.left - margin, bb.right + margin
    yt = bb.top + margin
    over = 0.5  # overlap so gmsh fuses the pieces

    walls = [
        (xl, yb, xl + wall + over, yt),  # left
        (xr - wall - over, yb, xr, yt),  # right
        (xl, yt - wall - over, xr, yt),  # top
        (xl, yb, xr, yb + wall + over),  # bottom, under the lead ends
    ]
    for x1, y1, x2, y2 in walls:
        c.add_polygon([(x1, y1), (x1, y2), (x2, y2), (x2, y1)], layer=layer)

    if shield:
        xi1, xi2 = xl + wall, xr - wall
        yi1, yi2 = yb + wall, yt - wall
        ym = 0.5 * (yi1 + yi2)
        pitch = shield_width + shield_space
        n = int((xi2 - xi1 - shield_space) / pitch)
        for i in range(n):
            x = xi1 + shield_space + i * pitch
            for y1, y2 in (
                (yi1 - over, ym - shield_space / 2),
                (ym + shield_space / 2, yi2 + over),
            ):
                c.add_polygon(
                    [(x, y1), (x, y2), (x + shield_width, y2), (x + shield_width, y1)],
                    layer=layer,
                )
    c.flatten()
    return c


cc = add_ground_frame(coil)
cc.plot()

In [ ]:
add_ground_frame(coil, shield=True).plot()

## 3. Palace simulation

Three `interlayer` lumped ports (Metal1 → TopMetal2), all excited, give the full 3×3 S-matrix referenced to 50 Ω.

In [ ]:
from gsim.palace import DrivenSim

PORTS = ("P1", "P2", "P3")
FMIN, FMAX = 1e9, 150e9


def make_sim(
    component,
    output_dir,
    preset="default",
    num_points=150,
    adaptive_tol=0.02,
    refined_mesh_size=3.0,
):
    """Configure a 3-port driven simulation of a framed T-coil."""
    sim = DrivenSim()
    sim.set_output_dir(output_dir)
    sim.set_geometry(component)
    sim.set_stack(substrate_thickness=180.0, include_substrate=True)
    for name in PORTS:
        sim.add_port(
            name,
            from_layer="metal1",
            to_layer="topmetal2",
            geometry="interlayer",
            excited=True,
        )
    sim.set_driven(
        fmin=FMIN, fmax=FMAX, num_points=num_points, adaptive_tol=adaptive_tol
    )
    sim.set_airbox(margin_x=50, margin_y=50, z_above=50, z_below=5)
    sim.mesh(preset=preset, refined_mesh_size=refined_mesh_size)
    sim.write_config()
    return sim


sim = make_sim(cc, "./palace-sim-tcoil")
print(sim.validate_config())

In [ ]:
sim.plot_mesh(show_groups=["metal", "P", "via"])

### Run simulation on cloud

In [ ]:
results = sim.run(check_cache=True)

In [ ]:
results.plot_interactive()

## 4. Extract the lumped T-coil from the EM data

**Quasi-static view.** Shorting the center tap to ground removes P3 from the nodal matrix; inverting the remaining (P1, P2) block gives $Z'_{11} = j\omega L_1$, $Z'_{22} = j\omega L_2$, $Z'_{12} = -j\omega M$ at low frequency, hence $k = M/\sqrt{L_1 L_2}$.

**Broadband fit.** The full 9-parameter model above is then fitted to the simulated S-matrix (least squares in log-parameter space, $0 < k < 1$), which also separates the bridge capacitance $C_B$ from the shunt parasitics.

In [ ]:
from scipy.optimize import least_squares


def sim_y(results):
    """Frequencies (Hz) and Y-matrix ordered (P1, P2, P3)."""
    ntwk = results.to_skrf()
    order = [list(results.port_names).index(p) for p in PORTS]
    Y = ntwk.y[:, order][:, :, order]
    return results.freq * 1e9, Y  # results.freq is in GHz


def low_freq_lmk(f, Y, fmax=20e9):
    """L1, L2, M, k and series R from the tap-grounded 2x2 impedance."""
    w = 2 * np.pi * np.asarray(f)
    Zp = np.linalg.inv(Y[:, :2, :2])
    sel = np.asarray(f) <= fmax
    L1 = np.median(Zp[sel, 0, 0].imag / w[sel])
    L2 = np.median(Zp[sel, 1, 1].imag / w[sel])
    M = -np.median(Zp[sel, 0, 1].imag / w[sel])
    R1 = np.median(Zp[sel, 0, 0].real)
    R2 = np.median(Zp[sel, 1, 1].real)
    return {"L1": L1, "L2": L2, "M": M, "k": M / np.sqrt(L1 * L2), "R1": R1, "R2": R2}


_NAMES = ("L1", "L2", "k", "C_B", "R1", "R2", "C_A", "C_Bg", "C_T")
_SCALE = np.array([1e-12, 1e-12, 1.0, 1e-15, 1.0, 1.0, 1e-15, 1e-15, 1e-15])


def fit_tcoil(f, Y, fmax=None):
    """Least-squares fit of the lumped T-coil to a simulated 3-port Y."""
    f = np.asarray(f)
    sel = slice(None) if fmax is None else f <= fmax
    S_ref = y_to_s(Y[sel])
    lf = low_freq_lmk(f, Y)
    x0 = {
        "L1": lf["L1"],
        "L2": lf["L2"],
        "k": float(np.clip(lf["k"], 0.05, 0.9)),
        "C_B": 5e-15,
        "R1": max(lf["R1"], 0.1),
        "R2": max(lf["R2"], 0.1),
        "C_A": 5e-15,
        "C_Bg": 5e-15,
        "C_T": 5e-15,
    }
    u0 = np.log(np.array([x0[n] for n in _NAMES]) / _SCALE)

    def unpack(u):
        return dict(zip(_NAMES, np.exp(u) * _SCALE, strict=True))

    def resid(u):
        d = (y_to_s(tcoil_y(f[sel], **unpack(u))) - S_ref).ravel()
        return np.concatenate([d.real, d.imag])

    hi = np.full(len(_NAMES), np.inf)
    hi[_NAMES.index("k")] = np.log(0.99)
    sol = least_squares(resid, u0, bounds=(-np.inf, hi))
    out = unpack(sol.x)
    out["M"] = out["k"] * np.sqrt(out["L1"] * out["L2"])
    out["rms_S"] = float(np.sqrt(np.mean(sol.fun**2)))
    return out


def fmt(params):
    """Pretty-print a parameter dict in pH / fF / Ohm."""
    unit = {"L": (1e12, "pH"), "M": (1e12, "pH"), "C": (1e15, "fF"), "R": (1, "Ohm")}
    parts = []
    for k, v in params.items():
        scale, u = unit.get(k[0], (1, ""))
        parts.append(f"{k}={v * scale:.3g}{u}")
    return ", ".join(parts)


f, Y = sim_y(results)
lf = low_freq_lmk(f, Y)
fit = fit_tcoil(f, Y, fmax=100e9)  # below the self-resonance
print("quasi-static:", fmt(lf))
print("broadband fit:", fmt(fit))
print("targets:", fmt(targets))

In [ ]:
S_sim = y_to_s(Y)
S_fit = y_to_s(tcoil_y(f, **{k: fit[k] for k in _NAMES}))
S_target = y_to_s(
    tcoil_y(f, targets["L1"], targets["L2"], targets["k"], targets["C_B"])
)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (i, j) in zip(axes, [(0, 0), (2, 0), (1, 0)], strict=True):
    ax.plot(f / 1e9, 20 * np.log10(np.abs(S_sim[:, i, j])), ".", ms=3, label="Palace")
    ax.plot(f / 1e9, 20 * np.log10(np.abs(S_fit[:, i, j])), label="Lumped fit")
    ax.plot(f / 1e9, 20 * np.log10(np.abs(S_target[:, i, j])), "--", label="Target")
    ax.set(xlabel="Frequency (GHz)", ylabel=f"|S{i + 1}{j + 1}| (dB)")
    ax.grid(True)
    ax.legend()
plt.tight_layout()
plt.show()

The fit uses frequency-independent $R_1, R_2$, so it slightly underestimates the loss at low frequency and overestimates it near the top of the band (skin effect makes the real series resistance grow with frequency). $L_1, L_2, k$ and $C_B$ are the quantities to compare with the targets: in this default layout $k$ comes out above $1/3$ and $C_B$ below $C_L/8$, which is why the sweep below varies the turn spacing as well as the diameter.

## 5. Matching-network performance

Close the EM 3-port as in the target circuit: P1 driven from 50 Ω, P2 terminated in 50 Ω (already the S-parameter reference) and P3 loaded with $C_L$. We read the return loss at P1 and the −3 dB bandwidth of the voltage delivered to $C_L$, and compare with the ideal T-coil and the bare load.

In [ ]:
RL_MIN_DB = 10.0  # return-loss target at P1 [dB]
F_SPEC = 60e9  # band over which the return loss must hold [Hz]
BW_MIN = 80e9  # bandwidth target into C_L [Hz]
AREA_MAX_UM2 = 80.0**2  # coil footprint limit [um^2]


def evaluate(f, S, area_um2):
    """Electrical + area metrics of a candidate against the declared targets."""
    S11, H = terminated_response(S, f, C_L)
    band = f <= F_SPEC
    rl = -20 * np.log10(np.abs(S11[band]).max())
    bw = bandwidth(f, H)
    return {
        "return_loss_dB": rl,
        "bw_GHz": bw / 1e9,
        "area_um2": area_um2,
        "pass_rl": bool(rl >= RL_MIN_DB),
        "pass_bw": bool(bw >= BW_MIN),
        "pass_area": bool(area_um2 <= AREA_MAX_UM2),
    }


S11_sim, H_sim = terminated_response(S_sim, f, C_L)
print(evaluate(f, S_sim, coil.info["area_um2"]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(f / 1e9, 20 * np.log10(np.abs(S11_sim)), label="Palace T-coil")
ax1.axhline(-RL_MIN_DB, color="gray", ls=":", label="Target")
ax1.axvline(F_SPEC / 1e9, color="gray", ls=":")
ax1.set(xlabel="Frequency (GHz)", ylabel="|S11| at P1 (dB)", title="Input match")
ax1.grid(True)
ax1.legend()

ax2.plot(f / 1e9, 20 * np.log10(np.abs(H_sim)), label="Palace T-coil")
ax2.plot(f_ideal / 1e9, 20 * np.log10(np.abs(H_ideal)), "--", label="Ideal T-coil")
ax2.plot(f_ideal / 1e9, 20 * np.log10(np.abs(H_bare)), ":", label="No T-coil")
ax2.axhline(-3, color="gray", lw=0.8)
ax2.set(
    xlabel="Frequency (GHz)",
    ylabel="|H| (dB)",
    title="Transfer to C_L",
    ylim=(-15, 3),
    xlim=(0, FMAX / 1e9),
)
ax2.grid(True)
ax2.legend()
plt.tight_layout()
plt.show()

## 6. Target-driven sweep with a fixed budget

A first, transparent search: a small grid over `diameter` (sets $L$) and `space` (sets $k$), filtered by the geometry constraints *before* any EM run, and evaluated in parallel on the cloud (non-blocking API, as in `palace_width_sweep`). The budget is the number of Palace evaluations. Every evaluated candidate is saved. Since all layouts of this family reach a similar bandwidth, the best one is chosen by the largest sum of normalized margins over return loss and bandwidth; if none is feasible, the candidate with the smallest total shortfall is reported as infeasible.

In [ ]:
import itertools
import json

import gsim

BUDGET = 9
grid = list(itertools.product([52.0, 60.0, 68.0], [2.0, 3.0, 4.0]))  # (diameter, space)
WIDTH = 3.0

candidates = [
    (d, s)
    for d, s in grid
    if not check_tcoil_constraints(d, WIDTH, s, max_area_um2=AREA_MAX_UM2)
][:BUDGET]
print(f"{len(candidates)} candidates within budget {BUDGET}: {candidates}")

sweep_sims = []
for d, s in candidates:
    comp = add_ground_frame(tcoil(diameter=d, width=WIDTH, space=s))
    sweep_sims.append(
        make_sim(comp, f"./palace-sim-tcoil-d{d:g}-s{s:g}", preset="default")
    )

job_ids = [s.run(wait=False, check_cache=True) for s in sweep_sims]
sweep_results = gsim.wait_for_results(job_ids)

In [ ]:
from pathlib import Path


def margin(r):
    """Sum of normalized margins over the electrical targets (higher is better)."""
    return (r["return_loss_dB"] - RL_MIN_DB) / RL_MIN_DB + (
        r["bw_GHz"] - BW_MIN / 1e9
    ) / (BW_MIN / 1e9)


records = []
for (d, s), res in zip(candidates, sweep_results, strict=True):
    f_c, Y_c = sim_y(res)
    lf_c = low_freq_lmk(f_c, Y_c)
    rec = {"diameter": d, "width": WIDTH, "space": s}
    rec.update({k: float(v) for k, v in lf_c.items()})
    rec.update(evaluate(f_c, y_to_s(Y_c), d**2))
    rec["feasible"] = rec["pass_rl"] and rec["pass_bw"] and rec["pass_area"]
    rec["margin"] = margin(rec)
    records.append(rec)

for r in records:
    print(
        f"d={r['diameter']:4.0f} s={r['space']:.0f}  L1={r['L1'] * 1e12:5.1f} pH  "
        f"k={r['k']:.2f}  RL={r['return_loss_dB']:5.1f} dB  BW={r['bw_GHz']:5.1f} GHz  "
        f"margin={r['margin']:+.2f}  {'PASS' if r['feasible'] else 'fail'}"
    )

feasible = [r for r in records if r["feasible"]]
if feasible:
    best = max(feasible, key=margin)
    print("\nBest feasible design (largest margin):", best)
else:
    # Margins can trade a large surplus in one target against a deficit in
    # another, so infeasible candidates are ranked by their total shortfall.
    def shortfall(r):
        return max(0, RL_MIN_DB - r["return_loss_dB"]) / RL_MIN_DB + max(
            0, BW_MIN / 1e9 - r["bw_GHz"]
        ) / (BW_MIN / 1e9)

    best = min(records, key=shortfall)
    print("\nINFEASIBLE within budget; closest candidate:", best)

report = {
    "pdk": "ihp-gdsfactory==0.2.8 (SG13G2)",
    "targets": {
        "R_T_ohm": R_T,
        "C_L_fF": C_L * 1e15,
        "zeta": float(ZETA),
        "return_loss_dB": RL_MIN_DB,
        "f_spec_GHz": F_SPEC / 1e9,
        "bw_min_GHz": BW_MIN / 1e9,
        "area_max_um2": AREA_MAX_UM2,
        "ideal_tcoil": {k: float(v) for k, v in targets.items()},
    },
    "budget": BUDGET,
    "selection": "largest sum of normalized margins over return loss and bandwidth",
    "candidates": records,
    "best": best,
}
REPORT_PATH = Path("./palace-sim-tcoil-sweep/report.json")
REPORT_PATH.parent.mkdir(exist_ok=True)
with open(REPORT_PATH, "w") as fh:
    json.dump(report, fh, indent=2, default=float)

### Sweep overview

Each point is one candidate layout, placed by its bandwidth to $C_L$ (x axis) and its return loss up to 60 GHz (y axis). The shaded area is where both targets are met; candidates outside it are shown in gray. Feasible candidates are colored by their margin, the sum of how much they exceed each target relative to that target, and the one with the largest margin (star) is the best design. The plot also shows the trade-off set by the turn spacing: tighter spacing ($s = 2$) gives a better match but less bandwidth, wider spacing ($s = 4$) the opposite.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

bw = [r["bw_GHz"] for r in records]
rl = [r["return_loss_dB"] for r in records]

# Feasible region: right of the BW target and above the RL target
ax.fill_between(
    [BW_MIN / 1e9, 200], RL_MIN_DB, 100, color="C2", alpha=0.08, label="Feasible region"
)
ax.axvline(
    BW_MIN / 1e9,
    color="gray",
    ls="--",
    lw=0.8,
    label=f"BW target: {BW_MIN / 1e9:g} GHz",
)
ax.axhline(
    RL_MIN_DB, color="gray", ls=":", lw=0.8, label=f"RL target: {RL_MIN_DB:g} dB"
)

# Candidates: gray if they fail a target, colored by margin if feasible
ok = [r for r in records if r["feasible"]]
ko = [r for r in records if not r["feasible"]]
ax.scatter(
    [r["bw_GHz"] for r in ko],
    [r["return_loss_dB"] for r in ko],
    color="lightgray",
    edgecolors="gray",
    s=60,
    zorder=3,
    label="Fails a target",
)
sc = ax.scatter(
    [r["bw_GHz"] for r in ok],
    [r["return_loss_dB"] for r in ok],
    c=[r["margin"] for r in ok],
    cmap="viridis",
    s=60,
    zorder=3,
)
fig.colorbar(sc, ax=ax, label="Margin (feasible designs)")

for r in records:
    ax.annotate(
        f"d={r['diameter']:g}, s={r['space']:g}",
        (r["bw_GHz"], r["return_loss_dB"]),
        textcoords="offset points",
        xytext=(6, -10),
        fontsize=8,
    )

ax.scatter(
    best["bw_GHz"],
    best["return_loss_dB"],
    marker="*",
    s=300,
    facecolors="none",
    edgecolors="red",
    zorder=4,
    label="Best design",
)

ax.set(
    xlabel="Bandwidth to C_L (GHz)",
    ylabel="Return loss up to 60 GHz (dB)",
    title="Sweep candidates",
    xlim=(min(bw) - 1, max(bw) + 1),
    ylim=(min(rl) - 1, max(rl) + 1),
)
ax.grid(True)
ax.legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

### Independent re-solve of the best design

The winner is re-simulated with a finer mesh (1.5 µm instead of 3 µm near the conductors, so the 2–3 µm gaps between turns are resolved), twice as many frequency points and a tighter adaptive tolerance, and compared with its sweep result. If $L$, $k$, $C_B$, return loss and bandwidth barely move, the sweep mesh was converged.

In [ ]:
d_b, s_b = best["diameter"], best["space"]
fine_sim = make_sim(
    add_ground_frame(tcoil(diameter=d_b, width=WIDTH, space=s_b)),
    f"./palace-sim-tcoil-fine-d{d_b:g}-s{s_b:g}-m1.5",
    preset="fine",
    num_points=301,
    adaptive_tol=1e-3,
    refined_mesh_size=1.5,
)
fine_results = fine_sim.run(check_cache=False)

In [ ]:
f_fine, Y_fine = sim_y(fine_results)
S_fine = y_to_s(Y_fine)
lf_fine = low_freq_lmk(f_fine, Y_fine)
fit_fine = fit_tcoil(f_fine, Y_fine, fmax=100e9)
f_d, Y_d = sim_y(sweep_results[candidates.index((d_b, s_b))])
fit_default = fit_tcoil(f_d, Y_d, fmax=100e9)
print("sweep (default) fit:", fmt(fit_default))
print("re-solve (fine) fit:", fmt(fit_fine))
m_fine = evaluate(f_fine, S_fine, d_b**2)

for label, r in (("sweep (default)", best), ("re-solve (fine)", {**lf_fine, **m_fine})):
    print(
        f"{label:16s} L1={r['L1'] * 1e12:5.1f} pH  k={r['k']:.3f}  "
        f"RL={r['return_loss_dB']:5.1f} dB  BW={r['bw_GHz']:5.1f} GHz"
    )
ok = m_fine["pass_rl"] and m_fine["pass_bw"] and m_fine["pass_area"]
print("Fine re-solve:", "PASS" if ok else "fail")

report["fine_resolve"] = {
    "preset": "fine",
    "num_points": 301,
    "adaptive_tol": 1e-3,
    "refined_mesh_size_um": 1.5,
    **{k: float(v) for k, v in lf_fine.items()},
    "C_B": float(fit_fine["C_B"]),
    **m_fine,
    "feasible": bool(ok),
}
with open(REPORT_PATH, "w") as fh:
    json.dump(report, fh, indent=2, default=float)

S11_f, H_f = terminated_response(S_fine, f_fine, C_L)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Input match
w_f = 2 * np.pi * f_fine
S11_bare = -(1j * w_f * C_L * R_T) / (2 + 1j * w_f * C_L * R_T)
ax1.plot(f_fine / 1e9, 20 * np.log10(np.abs(S11_f)), color="C0", label="Best design")
ax1.plot(
    f_fine / 1e9, 20 * np.log10(np.abs(S11_bare)), ":", color="C2", label="No T-coil"
)
ax1.axhline(
    -RL_MIN_DB,
    color="gray",
    ls=":",
    label=f"Target: below -{RL_MIN_DB:g} dB up to {F_SPEC / 1e9:g} GHz",
)
ax1.axvline(
    F_SPEC / 1e9,
    color="gray",
    ls="--",
    lw=0.8,
)
ax1.set(xlabel="Frequency (GHz)", ylabel="|S11| at P1 (dB)", title="Input match")

# Transfer to the load capacitance
ax2.plot(f_fine / 1e9, 20 * np.log10(np.abs(H_f)), label="Best design ")
ax2.plot(f_ideal / 1e9, 20 * np.log10(np.abs(H_ideal)), "--", label="Ideal T-coil")
ax2.plot(f_ideal / 1e9, 20 * np.log10(np.abs(H_bare)), ":", label="No T-coil")
ax2.axhline(-3, color="gray", lw=0.8)
ax2.axvline(
    BW_MIN / 1e9, color="gray", ls="--", lw=0.8, label=f"Target: {BW_MIN / 1e9:g} GHz"
)
ax2.set(
    xlabel="Frequency (GHz)",
    ylabel="|H| (dB)",
    title="Transfer to C_L",
    ylim=(-15, 3),
    xlim=(0, FMAX / 1e9),
)

for ax in (ax1, ax2):
    ax.grid(True)
    ax.legend()
plt.tight_layout()
plt.show()

## Next steps

- **Optimizer instead of a grid:** the bandwidth is capped near 80 GHz by the shunt parasitics, and $C_B$ stays near half its ideal value. A variable that sets $C_B$ (for example a MIM capacitor between P1 and P2) is needed before a Bayesian optimizer can do better than this grid.
- **Shield / fill variants:** rerun the winner with `add_ground_frame(coil, shield=True)` and with metal fill.
- **Sensitivity sweeps** (not IHP corner data): TopMetal2/TopMetal1 thickness, conductivity and oxide permittivity via `stack` overrides.
- **Asymmetric T-coil** ($L_1 \neq L_2$), as in Li et al. 2023, when the source is not matched (current-mode driver with $R_1 \neq R_2$).